# Lab: optical terminals and the event bus
satComTopology V5 models the **optical terminals** of every satellite: where each terminal is mounted, which neighbour it may point to, and its physical limits (range, elevation, angular velocity, ...). With the `advanced` optical model, every inter-satellite link (ISL) is checked against both of its terminals at each tick.

V5 also publishes every change of the network on an **event bus**: an emulator subscribes to it instead of polling the simulation.

In this lab you will:
1. build a small **IRIS²-like** constellation (one Walker Delta shell at 1 200 km), with **4 optical terminals** per satellite
2. create an `EventBus` and subscribe to the ISL events
3. at every `ISL_UPDATED` event, print the optical constraints seen by **both terminals** of the link, next to the terminal limits
4. record a whole orbit and check how close each terminal gets to its limits

In [ ]:
from concurrent.futures import ThreadPoolExecutor

from sat_com_application.event_bus.bus import EventBus
from sat_com_application.event_bus.events import EventType, ISLUpdatedEvent, ISLCreatedEvent, ISLRemovedEvent
from sat_com_builder.configuration_manager import BaseConfigurationManager
from sat_com_builder.models import SimulationProperty

## 1. A small IRIS²-like constellation with 4 optical terminals
[IRIS²](https://defence-industry-space.ec.europa.eu/eu-space/iris2-secure-connectivity_en) is the European secure connectivity constellation; its LEO layer flies at about 1 200 km. We build a small version of it: **10 orbital planes × 10 satellites**, inclined at 55°.

The altitude is given as a mean motion: at 1 200 km a satellite makes about **13.16 revolutions per day** (orbital period ≈ 109 min).

Each satellite gets 4 optical terminals, declared in `satellite_property.optical_inter_satellite_link_installations`. All of them are mounted at `elevation_angle_configuration=-90` and differ by their azimuth (0° = along the velocity):

| Terminal | Azimuth | Points to |
|---|---|---|
| `INTRA_PLANE_FORWARD` | 0° | the next satellite of the same plane |
| `INTRA_PLANE_BACKWARD` | -180° | the previous satellite of the same plane |
| `INTER_PLANE_LEFT` | 90° | a satellite of the neighbouring plane |
| `INTER_PLANE_RIGHT` | -90° | a satellite of the other neighbouring plane |

`optical_terminal_property` holds the **limits** of the terminal. A link is only kept while both terminals stay inside them:
- `maximum_range_distance_km`: maximum distance to the other satellite
- `max_azimuth_degrees`: how far the terminal can rotate away from its mounting azimuth
- `min_elevation_degrees` / `max_elevation_degrees`: how far it can look below / above the local horizontal plane
- `maximum_angular_velocity_degree_per_second`: how fast it can track the other satellite
- `los_central_angle_max_degrees`: maximum central angle between both satellites (line of sight)
- `minimum_contact_duration_seconds`: a link expected to last less is not worth establishing

Finally, `optical_inter_satellite_link_model="advanced"` asks the simulation to compute the optical metrics of every link at each tick (the default `basic` model only follows the addressing plan and computes the distance).

In [ ]:
TERMINAL_LIMITS = {
    "maximum_range_distance_km": 7800.0,
    "max_azimuth_degrees": 180.0,
    "min_elevation_degrees": -25.0,
    "max_elevation_degrees": 5.0,
    "maximum_angular_velocity_degree_per_second": 1.0,
    "los_central_angle_max_degrees": 80.0,
    "minimum_contact_duration_seconds": 30.0,
}


def optical_terminal(azimuth, descriptor):
    return {
        "azimuth_angle_configuration": azimuth,
        "elevation_angle_configuration": -90.0,
        "descriptor": descriptor,
        "orientation_possibilities": [descriptor],
        "optical_terminal_property": TERMINAL_LIMITS,
    }


lab_config = {
    "simulation_name": "Optical Terminal Lab",
    "start_date": "2026-01-01 00:00:00.000000",
    "end_date": "2026-01-01 01:50:00.000000",
    "movement_model": "pyorbital",
    "distance_model": "sklearn",
    "optical_inter_satellite_link_model": "advanced",
    "walker_shells": [
        {
            "type": "delta",
            "static_addressing_plan": "FULL_MESH_WALKER_DELTA",
            "satellite_property": {
                "optical_inter_satellite_link_installations": [
                    optical_terminal(0.0, "INTRA_PLANE_FORWARD"),
                    optical_terminal(-180.0, "INTRA_PLANE_BACKWARD"),
                    optical_terminal(90.0, "INTER_PLANE_LEFT"),
                    optical_terminal(-90.0, "INTER_PLANE_RIGHT"),
                ]
            },
            "constellation_property": {
                "identifier": "Mini IRIS2",
                "amount_of_orbit_plane": 10,
                "amount_of_satellite_per_orbit_plane": 10,
                "inclination": 55.0,
                "phase_difference_between_satellites": True,
                "mean_revolution_per_day": 13.16,
            },
            "orbital_connectivity_property": {"adjacent_inter_satellite_shifting": 0},
        }
    ],
}

simulation_properties = SimulationProperty(**lab_config)

## 2. Create the event bus and subscribe to ISL events
An event handler is an `async` function receiving one event. `ISLUpdatedEvent` carries the link (`link_id`, `satellite_a_id`, `satellite_b_id`, `direction`, `distance_km`) and, for an `advanced` link, the optical metrics:

| Field | Meaning |
|---|---|
| `terminal_a_id` / `terminal_b_id` | the terminal used at each end |
| `azimuth_a_deg` / `azimuth_b_deg` | pointing azimuth of each terminal |
| `relative_azimuth_a_deg` / `relative_azimuth_b_deg` | rotation away from its mounting azimuth (checked against `max_azimuth_degrees`) |
| `elevation_a_deg` / `elevation_b_deg` | pointing elevation (checked against `min/max_elevation_degrees`) |
| `angular_velocity_a_deg_s` / `angular_velocity_b_deg_s` | tracking speed (checked against `maximum_angular_velocity_degree_per_second`) |
| `range_rate` | how fast the distance changes, in km/s |
| `los_min_altitude_km` | lowest altitude of the line of sight (it must stay above the atmosphere) |
| `temporary_unavailable` / `recovery_time` | the link is down for now, and when it should come back |

Events only carry terminal ids. To print readable names and the limits, we build a lookup table `terminal_id -> terminal` once the simulation is loaded: every satellite lists its terminals in `optical_link_terminals`.

The `PRINT_EVENTS` switch lets us print every event during a short run, then only record them during a long one.

In [ ]:
terminals_by_id = {}
updated_events = []
created_count = {"value": 0}
PRINT_EVENTS = {"enabled": True}


def describe_terminal(side, terminal_id, azimuth, relative_azimuth, elevation, angular_velocity):
    terminal = terminals_by_id[terminal_id]
    limits = terminal.terminal_configuration
    return (
        f"    {side} {terminal.host_satellite.satellite_name:<14} {terminal.descriptor.value:<21}"
        f" az {azimuth:8.2f}° (rel {relative_azimuth:6.2f}° ≤ {limits.max_azimuth_degrees:.0f})"
        f"  el {elevation:7.2f}° ∈ [{limits.min_elevation_degrees:.0f}, {limits.max_elevation_degrees:.0f}]"
        f"  ω {angular_velocity:.5f}°/s ≤ {limits.maximum_angular_velocity_degree_per_second}"
    )


async def on_isl_updated(event: ISLUpdatedEvent):
    updated_events.append(event)
    if not PRINT_EVENTS["enabled"]:
        return

    status = " TEMPORARILY UNAVAILABLE" if event.temporary_unavailable else ""
    print(
        f"[{event.sim_time:%H:%M:%S}] {event.direction:<8} ISL {event.satellite_a_id:>3} <-> {event.satellite_b_id:<3}"
        f" range {event.distance_km:7.1f} km, range rate {event.range_rate:+.4f} km/s,"
        f" line of sight ≥ {event.los_min_altitude_km:.0f} km{status}"
    )
    print(describe_terminal("A", event.terminal_a_id, event.azimuth_a_deg, event.relative_azimuth_a_deg,
                            event.elevation_a_deg, event.angular_velocity_a_deg_s))
    print(describe_terminal("B", event.terminal_b_id, event.azimuth_b_deg, event.relative_azimuth_b_deg,
                            event.elevation_b_deg, event.angular_velocity_b_deg_s))


async def on_isl_created(event: ISLCreatedEvent):
    created_count["value"] += 1


async def on_isl_removed(event: ISLRemovedEvent):
    print(f"[{event.sim_time:%H:%M:%S}] ISL {event.satellite_a_id} <-> {event.satellite_b_id} REMOVED")


event_bus = EventBus()
event_bus.subscribe(EventType.ISL_UPDATED, on_isl_updated)
event_bus.subscribe(EventType.ISL_CREATED, on_isl_created)
event_bus.subscribe(EventType.ISL_REMOVED, on_isl_removed)

## 3. Load the simulation
The bus is given to the configuration manager. Subscribe **before** `load_simulation()`: the initial topology is published while it loads (`TO_CREATED` for every satellite, `ISL_CREATED` for every link).

> **Events in Jupyter.** Handlers are `async`. When the simulation runs inside an event loop, the `SimulationManager` *schedules* each event on it instead of running the handler right away. Jupyter always runs an event loop, so the handlers would only run once the cell is finished, all at once. To see each event as it happens, we run the simulation in a worker thread with `run_synchronously`: there is no event loop in that thread, so every handler runs immediately, in order. In a plain Python script you don't need it.

In [ ]:
def run_synchronously(function, *args, **kwargs):
    """Run `function` outside Jupyter's event loop, so the event bus calls each handler right away."""
    with ThreadPoolExecutor(max_workers=1) as executor:
        return executor.submit(function, *args, **kwargs).result()


configuration_manager = BaseConfigurationManager(simulation_property=simulation_properties, event_bus=event_bus)
simulation_manager = run_synchronously(configuration_manager.load_simulation)

for satellite in simulation_manager.get_satellites():
    for terminal in satellite.optical_link_terminals:
        terminals_by_id[str(terminal.terminal_id)] = terminal

print(f"{len(simulation_manager.get_satellites())} satellites, {len(terminals_by_id)} optical terminals")
print(f"{created_count['value']} ISL_CREATED events received while loading")

## 4. Print every ISL update
One tick of 60 seconds: every link is updated, so we receive one `ISL_UPDATED` event per link (200 of them). For each one we print the link, then terminal A and terminal B with their limits.

What to look at:
- **intra-plane** links (`orbital`): both satellites fly one behind the other. The forward terminal points at about 0° of azimuth, the backward one at about 180°, both slightly below the horizontal plane (the Earth curves away), and they barely move (ω ≈ 0).
- **inter-plane** links (`adjacent`): the planes cross, so the neighbour drifts. Azimuth and angular velocity change with the latitude, and the range breathes.

In [ ]:
run_synchronously(simulation_manager.time_manager.tick, 60)

## 5. Record a whole orbit
We now stop printing and only record the events, for one orbit (≈ 109 minutes, one tick per minute). Then, for each kind of terminal, we compare the worst values reached to the terminal limits.

In [ ]:
PRINT_EVENTS["enabled"] = False
updated_events.clear()

run_synchronously(simulation_manager.time_manager.tick_until_the_end, ticking_time_in_seconds=60)

print(f"{len(updated_events)} ISL_UPDATED events recorded")

In [ ]:
from collections import defaultdict

measures = defaultdict(lambda: defaultdict(list))

for event in updated_events:
    for terminal_id, relative_azimuth, elevation, angular_velocity in (
        (event.terminal_a_id, event.relative_azimuth_a_deg, event.elevation_a_deg, event.angular_velocity_a_deg_s),
        (event.terminal_b_id, event.relative_azimuth_b_deg, event.elevation_b_deg, event.angular_velocity_b_deg_s),
    ):
        descriptor = terminals_by_id[terminal_id].descriptor.value
        measures[descriptor]["range"].append(event.distance_km)
        measures[descriptor]["relative_azimuth"].append(abs(relative_azimuth))
        measures[descriptor]["elevation"].append(elevation)
        measures[descriptor]["angular_velocity"].append(angular_velocity)

limits = simulation_manager.get_satellites()[0].optical_link_terminals[0].terminal_configuration
print(
    f"Limits: range ≤ {limits.maximum_range_distance_km:.0f} km, |relative azimuth| ≤ {limits.max_azimuth_degrees:.0f}°, "
    f"elevation ∈ [{limits.min_elevation_degrees:.0f}°, {limits.max_elevation_degrees:.0f}°], "
    f"ω ≤ {limits.maximum_angular_velocity_degree_per_second}°/s\n"
)
print(f"{'terminal':<22}{'range (km)':>20}{'max |rel. az|':>15}{'elevation':>22}{'max ω (°/s)':>14}")
for descriptor, values in sorted(measures.items()):
    print(
        f"{descriptor:<22}"
        f"{min(values['range']):>9.0f} – {max(values['range']):<8.0f}"
        f"{max(values['relative_azimuth']):>14.1f}°"
        f"{min(values['elevation']):>11.1f}° – {max(values['elevation']):.1f}°"
        f"{max(values['angular_velocity']):>14.4f}"
    )

unavailable = sum(1 for event in updated_events if event.temporary_unavailable)
print(f"\n{unavailable} update(s) flagged as temporarily unavailable")

What the orbit tells us:
- **Intra-plane terminals are almost static**: the satellite ahead stays at the same distance (≈ 4 680 km), straight ahead and about 18° below the horizontal plane. Their angular velocity is close to 0.
- **Inter-plane terminals do all the work**: the range goes from a few hundred km (where two planes cross, at high latitude) to more than 6 000 km (at the equator). The elevation gets close to the -25° limit and the terminal has to track at up to ≈ 0.5°/s.
- **The relative azimuth reaches 180°**: with `max_azimuth_degrees=180` a terminal can turn all the way round, so a `LEFT` terminal can serve a neighbour on the right (look at the terminals of link `0 <-> 10` above). Real terminals have a narrower field of regard.

## What's next?
- **Break the constellation**: set `amount_of_orbit_plane` to 8. The planes get 45° apart and the inter-plane neighbours drop below the -25° elevation limit near the equator. Watch the loader report the links it could not establish, and the `ISL_REMOVED` events while the satellites move.
- **Tighten a terminal**: set `max_azimuth_degrees` to 90 (or lower `maximum_angular_velocity_degree_per_second`) on the `INTER_PLANE_*` terminals only, and see which links change terminal, go `temporary_unavailable` or get removed.
- `ISL_CREATED` events also carry `estimated_duration_a` / `estimated_duration_b`: how long each terminal expects to keep the link.
- Customise the events themselves with an **event factory**: [inter_shell_cesium_lab.ipynb](inter_shell_cesium_lab.ipynb) adds the satellite positions to every ISL event, and replays them in Cesium.